## Imports

In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import random
import time 

## Reading in Data

In [5]:
data = pd.read_csv('../Data/exon_ranges_summary.csv')

## Simulation function

Defaults creating a CSV to False. (so don't get a ton of .csv's)

work in progress! 

In [16]:
def simulation(data, simulation_rounds=100, num_active_te_index=0, te_mobilize_threshold=0.5, to_csv=False):
    mean_length = 5000 # avg. TE length
    std_dev_length = 2000 # std. dev. of TE length

    # num_TES = 100
    active_tes = [1000, 10000, 10000]

    te_lengths = np.random.normal(loc=mean_length, scale=std_dev_length, size=active_tes[num_active_te_index])
    

    te_lengths = np.clip(te_lengths, 100, 10000).astype(int)
    print(te_lengths)

    genome_size = data['Genome_size']
    average_range = data['Average Range']

    range_1_start, range_1_end = 0, average_range
    range_2_start, range_2_end = average_range + 1, genome_size

    # Dictionary to store the results
    results = {
        'Species': data['Species'],
        'Beginning_GENOME_SIZE': genome_size,
        'Active_tes': active_tes[num_active_te_index],
        'range_1_start': range_1_start,
        'range_1_end': range_1_end,
        'range_2_start': range_2_start,
        'range_2_end': range_2_end,
        'TE_mobilized': 0,
        'TE_static': 0,
        'TE_in_exons': 0,
        'TE_in_non_coding': 0,
        'Exon_new_size': range_1_end - range_1_start + 1,
        'Non_coding_new_size': range_2_end - range_2_start + 1,
        'Total_Genome_growth': 0
    }

    for _ in range(simulation_rounds): # or active Te's ? 
        for _ in range(active_tes[num_active_te_index]):
            te_length = random.choice(te_lengths) # randomly select a TE length from normal distribution
            if random.random() < te_mobilize_threshold:
                prob_exon = range_1_end / genome_size # probability of TE landing in exon
                # te_position = random.randint(0, genome_size)
                # print(f'TE position: {te_position}')

                if random.random() < prob_exon: # lands in exon (range 1)
                    range_1_end += te_length # expand range 1
                    range_2_start += range_1_end + 1 # adjust range 2 start
                    range_2_end += te_length # expand range 2 
                    results['TE_in_exons'] += 1
                else:
                    range_2_end += te_length # expand range 2
                    results['TE_in_non_coding'] += 1 # increment count for non-coding TEs
        
                results['TE_mobilized'] += 1
            else:
                results['TE_static'] += 1

    results['Exon_new_size'] = range_1_end - range_1_start + 1
    results['Non_coding_new_size'] = range_2_end - range_2_start + 1
    results['Total_Genome_growth'] = results['Exon_new_size'] + results['Non_coding_new_size'] - genome_size
    
    results_df = pd.DataFrame(results, index=[0])
    if to_csv:
        results_df.to_csv('CSV/simulation_results.csv', index=False)
    return results_df, te_lengths

## Run simulation for Sparrow Hawk. 

can do other species or all species, just change what is being passed to function. Change csv to true below if you would like to see the .csv

In [17]:
sparrow_hawk_df, sparrow_te_lengths = simulation(data.iloc[0], to_csv=False)
sparrow_hawk_df

[ 2599  5502  5393  2624  8392  5218  1831  3732  9708  4650  3000  5536
  3482  5154  5030  6126  4277  8136  5866  3183  3157  5833  5041  7775
  6595  5307  5626  3418  7375  7642  6225  1734  8502  3372  5565  3469
  5851  3174  3855  4625  5337  4294  6391  2646  5009  6640  8261  7441
  7157  7822   362  1490  6307  7260  7318  5031   683  3497  4286  6660
  2857  3551  5401  8264  6363  2707  4016  9573  6072  4320  5720  6959
  7463  4438  7139  3291  5647  3807  6165  6852  6359  1566  2374  9613
  4482  7830  5489  1066  4631  5504  7712  4997  7059  3225  5681  6080
  2594  7361  3943  3245  5436  6069  4156  3963  8094  3129  2759  3943
  6895 10000  6724  7966  5033  6289  1868  6903  5481  4553  9037  4712
  6865  4531  6263  3033  1288  7350  5125  3528  3890  4949  5335  5112
  3813  5481  6682  7644  3032  4664  5517   652  3816  4254  4423  5328
  5539  5144  2338  6956  2820  5539  5131  2412  7063  2712  5498  5754
  6110  3983  3883  8333  3972  5892  7228  2400  6

,Species,Beginning_GENOME_SIZE,Active_tes,range_1_start,range_1_end,range_2_start,range_2_end,TE_mobilized,TE_static,TE_in_exons,TE_in_non_coding,Exon_new_size,Non_coding_new_size,Total_Genome_growth
0,Accipiter_nisus.Accipiter_nisus_ver1.0.112,1190649881,1000,0,533365,533366,1190649881,50027,49973,26,50001,661967,1432312135,242324221


## View TE lengths

In [24]:
np.set_printoptions(threshold=np.inf)
print(sparrow_te_lengths)


[ 3708  4962  6869  7200  2180  6570  5502  5473  7535  6015  5363  8229
  8342  2774  6942  5559  7329  3051  2437  6130  3344  5278  6688  6613
  4445  4807   100  8413  5022  3833  3450  7375  4868  8018  5121  2584
  4323  6466  5253  7545  4904  2741  5251  4088  7970  4400  6670  7446
  7999  4162  6098  6057  6052  6240  5783  5539  4933  3696  4732  6488
  4698  4007  6780  6434  5409  3565  4079  4319  3791  4228  5675  5568
   100  3269  6671  6161  5858  6379  4629  4182  6137  8171  1386  1612
  3294  5373  5761  8766  4474  6956  6184  7007  4037  5391  4482  6680
  4266  5330   827  5845  6292  5748  3987  4622  2352  6862  4469  1121
   962  1878  5510  3919  4342  7271  7743  7125  5798  7320  5774  1550
  8105  5947  6808  4040  7868  3893  7521  3958  7187  4169  4635  1168
  6267  9403  6385  3448  3985  4322  4033  7492  7011  3587  8409  4142
  4449  3995  6321  3335  4256  4417  5599  6981  8316  4973  5024  4050
  6252  5757  4389  3214  3287  2977  4098  8230  4